# 노트북 1. Gemini 직접 호출 vs LangChain 호출 비교

> Phase 1 — 기초: LLM과 대화하는 법

같은 Gemini 모델인데, 부르는 방법이 두 가지입니다.
차이를 알아야 언제 뭘 쓸지 판단할 수 있습니다.

**학습 목표**
- google-genai SDK로 Gemini API를 직접 호출할 수 있다
- LangChain을 통해 동일한 Gemini 모델을 호출할 수 있다
- 두 방식의 반환 객체와 사용 패턴 차이를 설명할 수 있다
- LCEL(LangChain Expression Language) 체인의 기본 구조를 이해한다

**구성**
| Part | 내용 | 형태 |
|------|------|------|
| Part 1 — 이론 | google-genai vs LangChain 비교 + LCEL 기초 | 읽고 실행 |
| Part 2 — 실습 | API 호출 + 반환 객체 비교 + LCEL 체인 구성 | 직접 작성 |
| Part 3 — 챌린지 | 두 방식의 응답 시간 측정 비교 | 강사와 함께 |

In [ ]:
# 환경 설정 — 이 셀을 먼저 실행하세요
!pip install -q google-genai langchain-google-genai

import os
from google import genai

# API 키 설정 (Colab 환경 기준)
# 왼쪽 키 아이콘 → GEMINI_API_KEY 등록 후 실행
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)
print("환경 설정 완료")

환경 설정 완료


---

# Part 1 — 이론

개념을 마크다운 설명과 실행 가능한 코드 예시로 배웁니다.
읽고 실행하면서 따라와 주세요.

## 1.1 google-genai SDK 소개

Gemini API를 호출하는 가장 기본적인 방법은 Google이 공식 제공하는 **google-genai** SDK를 사용하는 것입니다.

> 2025년 11월부터 기존 `google-generativeai` 패키지는 지원 종료(EOL)되었습니다.
> 현재 공식 SDK는 `google-genai`이며, `Client` 패턴을 사용합니다.
> 인터넷에서 구버전(`import google.generativeai as genai`) 코드를 보게 될 수 있지만,
> 이 과정에서는 새 SDK만 사용합니다.

참고: https://docs.cloud.google.com/vertex-ai/generative-ai/docs/sdks/overview

기본 구조는 다음과 같습니다:

```python
from google import genai

client = genai.Client(api_key="...")
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="질문 내용",
)
```

google-genai SDK는 Google이 제공하는 1차 인터페이스입니다.
Gemini의 모든 기능(Live API, 이미지 생성 등)에 가장 먼저 접근할 수 있다는 것이 핵심 장점입니다.

아래 코드는 google-genai SDK로 간단한 질문을 보내고 응답을 받는 과정을 보여줍니다.

In [ ]:
# google-genai SDK로 Gemini 직접 호출
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="대한민국의 수도는 어디인가요?",
)
print(response.text)

대한민국의 수도는 서울입니다.



반환된 `response` 객체의 타입과 구조를 살펴봅시다.
API 응답에 어떤 정보가 담겨 있는지 아는 것이 중요합니다.

참고: 추론 토큰 수가 제공되는데 이에 대해선 추후 배우기로 합니다.

https://firebase.google.com/docs/ai-logic/count-tokens?hl=ko

In [ ]:
# 반환 객체 타입 확인
from pprint import pprint
print(f"타입: {type(response).__name__}")
print(f"텍스트: {response.text}")

# 사용된 토큰 정보
print(f"입력 토큰 수: {response.usage_metadata.prompt_token_count}")
print(f"추론 토큰 수: {response.usage_metadata.thoughts_token_count}")
print(f"출력 토큰 수: {response.usage_metadata.candidates_token_count}")

print(f"총 토큰 수: {response.usage_metadata.total_token_count}")

타입: GenerateContentResponse
텍스트: 대한민국의 수도는 서울입니다.

입력 토큰 수: 9
추론 토큰 수: None
출력 토큰 수: 9
총 토큰 수: 18


> 핵심: google-genai SDK의 반환 객체는 `GenerateContentResponse`입니다.
> `.text`로 텍스트를 꺼내고, `.usage_metadata`로 토큰 사용량을 확인할 수 있습니다.
> 토큰에 대해서는 노트북 4에서 자세히 다룹니다.

`contents`에는 단순 문자열뿐 아니라 리스트 형태로 여러 파트를 전달할 수도 있습니다.
이 구조는 이후 멀티턴 대화(노트북 3)와 이미지 입력에서 활용됩니다.

아래 코드는 `contents`를 리스트로 전달하는 방식을 보여줍니다.

In [ ]:
# contents를 리스트로 전달하는 방식
# 단일 문자열과 동일한 결과이지만, 구조를 명시적으로 보여줌
response_list = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=["대한민국에서 가장 높은 산은 무엇인가요?"],
)

print(response_list.text)

대한민국에서 가장 높은 산은 **한라산(Hallasan)**입니다.

*   **위치:** 제주도
*   **높이:** 약 1,947미터

한라산은 휴화산으로, 아름다운 자연경관과 백록담이라는 분화구 호수를 가지고 있어 유네스코 세계유산이자 국립공원으로 지정되어 있습니다.


### config 파라미터

`generate_content()`에는 `config` 딕셔너리를 전달하여 모델의 동작을 제어할 수 있습니다.
각 파라미터의 상세한 의미는 노트북 5(생성 파라미터)에서 다루지만,
여기서 config의 기본 형태를 미리 확인해둡니다.

아래 코드는 `temperature`와 `max_output_tokens`를 설정하는 예시를 보여줍니다.

In [ ]:
# config로 생성 파라미터 제어
# 교안 준비 시엔, 인공지능은 우리 삶과 지능의 모든 측면에 깊이 스며들어, 인류의 미래를 근본적으로 재편할 것입니다. 이렇게 나오네요
response_config = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="인공지능의 미래를 한 문장으로 예측해주세요.",
    config={
        "temperature": 0.0,         # 낮을수록 일관된 응답
        "max_output_tokens": 1024,   # 출력 길이 제한
    },
)

print(response_config.text)
print(f"\n출력 토큰 수: {response_config.usage_metadata.candidates_token_count}")

인공지능은 인간의 능력을 확장하고 사회 전반을 재편하며, 협력과 윤리적 성찰이 필수적인 새로운 시대를 열 것입니다.

출력 토큰 수: 36


### 사용 가능한 모델 목록 확인

어떤 모델이 사용 가능한지 API를 통해 직접 확인할 수 있습니다.

아래 코드는 현재 사용 가능한 Gemini 모델 목록 중 일부를 출력합니다.

In [ ]:
# 사용 가능한 모델 목록 확인
for m in client.models.list():
    # gemini 모델만 필터링하여 상위 10개 출력
    if "gemini" in m.name:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3-pro-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/gemini-embedding-001
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025


## 1.2 LangChain 래핑

**LangChain**은 LLM 애플리케이션 개발을 위한 프레임워크입니다.
다양한 LLM 제공자(Google, OpenAI, Anthropic 등)를 **통일된 인터페이스**로 다룰 수 있게 해줍니다.

Gemini를 LangChain으로 호출하려면 `langchain-google-genai` 패키지의 `ChatGoogleGenerativeAI` 클래스를 사용합니다.

```python
# Google Gemini
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# OpenAI → 클래스만 바꾸면 나머지 코드 동일
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4o")

# Anthropic
from langchain_anthropic import ChatAnthropic
model = ChatAnthropic(model="claude-sonnet-4-20250514")

# 아래 코드는 위 어떤 model이든 동일하게 동작
response = model.invoke("대한민국의 수도는?")
print(response.content)
```

LangChain을 통해 호출하면 `.invoke()`, `.stream()`, `.batch()` 같은 통일된 메서드를 사용할 수 있습니다.
가장 큰 가치는 **모델 교체 용이성**입니다. Google에서 OpenAI로 바꿔도 나머지 코드가 동일하게 동작합니다.

> `langchain-google-genai` 4.x부터 내부적으로 `google-genai` SDK를 사용합니다.
> 즉 LangChain은 google-genai SDK를 한 번 더 감싼 **래퍼(wrapper)**입니다.

아래 코드는 LangChain으로 Gemini를 호출하는 과정을 보여줍니다.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# LangChain으로 Gemini 호출
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=GEMINI_API_KEY,
)

response_lc = model.invoke("대한민국의 수도는 어디인가요?")
print(response_lc.content)

대한민국의 수도는 **서울**입니다.


### LangChain 메시지 타입

LangChain은 LLM과의 대화를 **메시지 객체**로 관리합니다.
주요 메시지 타입은 다음 세 가지입니다:

| 타입 | 역할 | 설명 |
|------|------|------|
| `HumanMessage` | 사용자 | 사용자가 보내는 메시지 |
| `AIMessage` | AI | 모델이 생성한 응답 |
| `SystemMessage` | 시스템 | 모델의 행동 규칙 (노트북 2에서 상세 학습) |

`.invoke()`에 문자열을 전달하면 내부적으로 `HumanMessage`로 변환됩니다.
메시지 객체를 직접 전달하는 것도 가능합니다.

아래 코드는 메시지 객체를 직접 생성하여 전달하는 방식을 보여줍니다.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

# 메시지 객체를 직접 생성하여 전달
messages = [
    SystemMessage(content="당신은 친절한 한국어 도우미입니다. 간결하게 답변하세요."),
    HumanMessage(content="파이썬이란 무엇인가요?"),
]

response_msg = model.invoke(messages)
print(f"타입: {type(response_msg).__name__}")
print(f"응답: {response_msg.content}")

타입: AIMessage
응답: 파이썬은 배우기 쉽고 다양한 분야에서 활용되는 강력한 프로그래밍 언어입니다.


### .stream() — 스트리밍 호출

LangChain은 `.stream()` 메서드를 제공합니다.
스트리밍은 응답을 한 번에 받는 대신, 토큰이 생성되는 즉시 하나씩 받아보는 방식입니다.
(스트리밍의 상세한 원리는 노트북 6에서 다룹니다.)

아래 코드는 `.stream()`으로 응답을 청크 단위로 받는 모습을 보여줍니다.

In [ ]:
# LangChain 스트리밍 호출
# 각 chunk는 AIMessageChunk 타입
for chunk in model.stream("파이썬의 장점을 3가지 알려주세요"):
    print(chunk.content, end="", flush=True)

print()  # 줄바꿈

파이썬의 장점은 매우 많지만, 핵심적인 3가지를 꼽자면 다음과 같습니다.

1.  **직관적이고 배우기 쉬운 문법 (높은 가독성)**
    *   다른 프로그래밍 언어에 비해 문법이 간결하고 직관적입니다. 마치 영어를 읽는 것과 같아서 초보자도 빠르게 학습할 수 있으며, 코드 가독성이 높아 여러 사람이 함께 작업하거나 나중에 코드를 유지보수할 때도 유리합니다. 이는 개발 시간을 단축시키고 오류 발생 가능성을 줄여줍니다.

2.  **높은 범용성과 활용성**
    *   웹 개발(Django, Flask), 데이터 과학 및 인공지능(NumPy, Pandas, TensorFlow, PyTorch), 자동화 스크립트, 게임 개발, 임베디드 시스템, 데스크톱 애플리케이션 등 매우 광범위한 분야에서 활용됩니다. 하나의 언어로 다양한 작업을 수행할 수 있다는 것이 큰 장점이며, 이는 파이썬 개발자가 여러 분야에서 활약할 수 있는 기반이 됩니다.

3.  **풍부한 라이브러리와 활발한 커뮤니티**
    *   파이썬은 방대한 수의 라이브러리(예: 웹 개발용 Django, Flask, 데이터 분석용 Pandas, NumPy, 머신러닝용 TensorFlow, scikit-learn 등)를 제공하여 개발 시간을 단축하고 효율성을 높여줍니다. 필요한 기능을 직접 구현할 필요 없이 이미 만들어진 라이브러리를 가져다 쓸 수 있습니다. 또한, 전 세계적으로 활발한 개발자 커뮤니티가 있어 문제 발생 시 도움을 받거나 정보를 얻기 쉽다는 점도 큰 강점입니다.


스트리밍에서 반환되는 각 청크의 타입을 확인해봅시다.

In [ ]:
# 스트리밍 청크 타입 확인
chunks = list(model.stream("파이썬의 장점을 3가지 알려주세요"))

print(f"총 청크 수: {len(chunks)}")
print(f"첫 번째 청크 타입: {type(chunks[0]).__name__}")
print(f"첫 번째 청크 내용: '{chunks[0].content}'")
print(f"마지막 청크 내용: '{chunks[-1].content}'")

총 청크 수: 8
첫 번째 청크 타입: AIMessageChunk
첫 번째 청크 내용: '파이썬의 주요 장점 3가지를 말씀드릴게요.

1.  **직관적이고 간결한 문법 ('
마지막 청크 내용: ''


### .batch() — 여러 입력을 한 번에 처리

`.batch()`는 여러 입력을 리스트로 전달하여 한 번에 처리합니다.
내부적으로 병렬 호출을 시도하므로, 여러 질문을 순차적으로 보내는 것보다 효율적일 수 있습니다.

아래 코드는 3개의 질문을 `.batch()`로 한 번에 처리하는 예시를 보여줍니다.

In [ ]:
# 여러 질문을 한 번에 처리
questions = [
    "대한민국의 수도는?",
    "일본의 수도는?",
    "프랑스의 수도는?",
]

responses = model.batch(questions)

for q, r in zip(questions, responses):
    print(f"Q: {q}")
    print(f"A: {r.content}")
    print()

Q: 대한민국의 수도는?
A: 대한민국의 수도는 **서울**입니다.

Q: 일본의 수도는?
A: 일본의 수도는 **도쿄**입니다.

Q: 프랑스의 수도는?
A: 프랑스의 수도는 **파리**입니다.



> 핵심: LangChain의 `ChatGoogleGenerativeAI`는 `.invoke()`, `.stream()`, `.batch()`라는
> 세 가지 호출 메서드를 제공합니다. 이 인터페이스는 모든 LangChain 모델에서 동일합니다.
> 즉 모델을 교체해도 호출 방식을 바꿀 필요가 없습니다.

## 1.3 반환 객체 비교: GenerateContentResponse vs AIMessage

두 방식의 가장 눈에 띄는 차이는 **반환 객체의 타입**입니다.

| 구분 | google-genai | LangChain |
|------|-------------|----------|
| 반환 타입 | `GenerateContentResponse` | `AIMessage` |
| 텍스트 접근 | `.text` | `.content` |
| 토큰 정보 | `.usage_metadata` | `.usage_metadata` + `.response_metadata` |
| 스트리밍 반환 | chunk별 `GenerateContentResponse` | `AIMessageChunk` |
| 모델 정보 | `.model_version` | `.response_metadata["model_name"]` |

아래 코드는 동일한 질문에 대한 두 반환 객체를 나란히 비교합니다.

In [ ]:
# 동일한 질문으로 양쪽 모두 호출
question = "1 + 1 = ?"

# 1. google-genai
resp_genai = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=question,
)

# 2. LangChain
resp_langchain = model.invoke(question)

print("=== google-genai ===")
print(f"  타입: {type(resp_genai).__name__}")
print(f"  텍스트 접근: .text → '{resp_genai.text}'")
print()

print("=== LangChain ===")
print(f"  타입: {type(resp_langchain).__name__}")
print(f"  텍스트 접근: .content → '{resp_langchain.content}'")

=== google-genai ===
  타입: GenerateContentResponse
  텍스트 접근: .text → '1 + 1 = 2'

=== LangChain ===
  타입: AIMessage
  텍스트 접근: .content → '2'


### google-genai 반환 객체 상세 구조

`GenerateContentResponse`에는 `candidates`라는 리스트가 포함되어 있습니다.
각 candidate는 모델이 생성한 하나의 응답 후보입니다.
일반적으로 1개의 candidate만 반환됩니다.

아래 코드는 candidates 구조를 살펴봅니다.

In [ ]:
# candidates 구조 확인
print(f"candidates 수: {len(resp_genai.candidates)}")

candidate = resp_genai.candidates[0]
print(f"finish_reason: {candidate.finish_reason}")
print(f"content parts 수: {len(candidate.content.parts)}")
print(f"첫 번째 part 텍스트: {candidate.content.parts[0].text}")

candidates 수: 1
finish_reason: FinishReason.STOP
content parts 수: 1
첫 번째 part 텍스트: 1 + 1 = 2


### LangChain AIMessage 상세 구조

LangChain의 `AIMessage`에는 `response_metadata` 딕셔너리와 `usage_metadata` 딕셔너리가 포함되어 있습니다.
여기에 토큰 사용량, 모델 정보, 안전 필터 등의 정보가 들어있습니다.

아래 코드는 AIMessage의 주요 속성들을 확인합니다.

In [ ]:
# AIMessage 주요 속성
print("=== AIMessage 기본 속성 ===")
print(f"  content: {resp_langchain.content}")
print(f"  type: {resp_langchain.type}")      # 'ai'
print(f"  id: {resp_langchain.id}")
print()

print("=== response_metadata ===")
for key, value in resp_langchain.response_metadata.items():
    print(f"  {key}: {value}")
print()

print("=== usage_metadata ===")
for key, value in resp_langchain.usage_metadata.items():
    print(f"  {key}: {value}")

=== AIMessage 기본 속성 ===
  content: 2
  type: ai
  id: lc_run--019c7392-50d7-7270-8893-840be9bc794c-0

=== response_metadata ===
  finish_reason: STOP
  model_name: gemini-2.5-flash
  safety_ratings: []
  model_provider: google_genai

=== usage_metadata ===
  input_tokens: 7
  output_tokens: 63
  total_tokens: 70
  input_token_details: {'cache_read': 0}
  output_token_details: {'reasoning': 62}


### 토큰 사용량 비교

양쪽 모두 토큰 사용량 정보를 제공하지만, 접근 방식이 다릅니다.
아래 코드는 동일 응답의 토큰 정보를 양쪽에서 추출하여 비교합니다.

추론 토큰에 대해선, langchain 은 default로 비활성화인 것이 눈에 띄는 군요.

In [ ]:
# 토큰 사용량 비교
print("=== 토큰 사용량 비교 ===")
print()

# google-genai
genai_usage = resp_genai.usage_metadata
print("google-genai:")
print(f"  입력: {genai_usage.prompt_token_count}")
print(f"  출력: {genai_usage.candidates_token_count}")
print(f"  합계: {genai_usage.total_token_count}")
print()

# LangChain
lc_usage = resp_langchain.usage_metadata
print("LangChain:")
print(f"  입력: {lc_usage.get('input_tokens')}")
print(f"  출력: {lc_usage.get('output_tokens')}")
print(f"  합계: {lc_usage.get('total_tokens')}")

=== 토큰 사용량 비교 ===

google-genai:
  입력: 7
  출력: 7
  합계: 35

LangChain:
  입력: 7
  출력: 63
  합계: 70


> 핵심: 같은 모델, 같은 질문이므로 토큰 수는 거의 동일합니다.
> 차이는 접근 방식뿐입니다.
> google-genai는 `prompt_token_count`/`candidates_token_count` 속성을,
> LangChain은 `input_tokens`/`output_tokens` 키를 사용합니다.

In [ ]:
from pprint import pprint
pprint(resp_genai.usage_metadata)

GenerateContentResponseUsageMetadata(
  candidates_token_count=7,
  prompt_token_count=7,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=7
    ),
  ],
  thoughts_token_count=21,
  total_token_count=35
)


### google-genai vs LangChain: 언제 무엇을 쓸까?

정리하면 다음과 같습니다:

| 상황 | 권장 방식 | 이유 |
|------|----------|------|
| Gemini 고유 기능 사용 (Live API, 이미지 생성 등) | google-genai | LangChain이 아직 래핑하지 않은 기능 |
| 빠른 프로토타이핑 | google-genai | 의존성 적고 직관적 |
| 체이닝, 메모리, 도구 등 조합 | LangChain | 오케스트레이션 기능 풍부 |
| 모델 교체 가능성 있음 | LangChain | 인터페이스 통일 |
| 프로덕션 에이전트 | LangChain + LangGraph | 상태 관리, 조건 분기 지원 |

> 실무에서는 LangChain/LangGraph를 메인으로 사용하되,
> 특수 기능이 필요할 때 google-genai를 직접 사용하는 패턴이 일반적입니다.

## 1.4 LCEL 맛보기

**LCEL**(LangChain Expression Language)은 LangChain의 핵심 패턴입니다.
여러 컴포넌트를 `|` 연산자(파이프)로 연결하여 하나의 **체인(chain)**을 구성합니다.

가장 기본적인 체인 구조는 다음과 같습니다:

```
PromptTemplate  |  ChatModel  |  OutputParser
     (입력 가공)       (LLM 호출)      (출력 추출)
```

- **PromptTemplate**: 입력 변수를 받아 프롬프트를 생성합니다
- **ChatModel**: 프롬프트를 LLM에 전달하고 응답을 받습니다
- **OutputParser**: 응답에서 필요한 부분만 추출합니다

이 패턴은 이후 노트북에서 계속 사용하게 되므로, 여기서 기초를 다져둡니다.

### 기본 체인 구성

아래 코드는 가장 간단한 LCEL 체인을 구성하고 실행하는 전체 과정을 보여줍니다.

| (파이프) 연산자는 LangChain의 Runnable 인터페이스에 정의된 __or__로, Runnable끼리 연결하면 RunnableSequence가 됩니다.

참고: https://reference.langchain.com/python/langchain_core/runnables/

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 프롬프트 템플릿 정의
# {topic}은 나중에 실제 값으로 대체될 변수
prompt = ChatPromptTemplate.from_template(
    "{topic}에 대해 {num} 개의 문장으로 설명해주세요."
)

# 2. 출력 파서 — AIMessage에서 텍스트(str)만 추출
output_parser = StrOutputParser()

# 3. 체인 구성: prompt → model → parser
chain = prompt | model | output_parser

# 4. 체인 실행
result = chain.invoke({"topic": "인공지능", "num": "2"})
print(result)
print(f"\n반환 타입: {type(result).__name__}")  # str

인공지능은 컴퓨터가 인간처럼 학습하고 추론하며 문제를 해결하는 능력을 갖추도록 하는 기술입니다. 이 기술은 복잡한 작업을 자동화하고, 데이터 기반의 의사결정을 돕는 등 다양한 분야에서 활용됩니다.

반환 타입: TextAccessor


`StrOutputParser`가 있으면 `AIMessage` 대신 순수 문자열(`str`)을 반환합니다.
각 단계가 이전 단계의 출력을 받아 다음 단계의 입력으로 전달하는 것이 핵심입니다.

```
{"topic": "인공지능"}  →  prompt  →  "인공지능에 대해 한 문장으로 설명해주세요."
                                          ↓
                                        model  →  AIMessage(content="인공지능이란...")
                                                       ↓
                                                  output_parser  →  "인공지능이란..."
```

LangChain의 주요 Output Parser들입니다:

**텍스트 기반**

| Parser | 용도 | 출력 타입 |
|--------|------|----------|
| `StrOutputParser` | AIMessage → 순수 문자열 추출 | `str` |
| `CommaSeparatedListOutputParser` | 쉼표 구분 텍스트 → 리스트 | `list[str]` |
| `NumberedListOutputParser` | 번호 매긴 목록 → 리스트 | `list[str]` |
| `MarkdownListOutputParser` | 마크다운 리스트 → 리스트 | `list[str]` |

**구조화 출력 (가장 많이 사용)**

| Parser | 용도 | 출력 타입 |
|--------|------|----------|
| `JsonOutputParser` | JSON 문자열 → dict/Pydantic | `dict` or `BaseModel` |
| `PydanticOutputParser` | Pydantic 모델로 강제 파싱 | `BaseModel` |


ChatPromptTemplate.from_template()으로 만들면 기본적으로 **HumanMessage**가 됩니다.

In [ ]:
prompt = ChatPromptTemplate.from_template("{topic}에 대해 {num} 개의 문장으로 설명해주세요.")
messages = prompt.format_messages(topic="AI", num="2")
print(messages)


[HumanMessage(content='AI에 대해 2 개의 문장으로 설명해주세요.', additional_kwargs={}, response_metadata={})]


### 체인 스트리밍

체인도 `.stream()`을 지원합니다. 개별 모델 호출과 마찬가지로 토큰 단위로 결과를 받아볼 수 있습니다.

아래 코드는 체인의 스트리밍 실행을 보여줍니다.

In [ ]:
# LCEL 체인 스트리밍
for token in chain.stream({"topic": "머신러닝", "num":"2"}):
    print(token, end="", flush=True)

print()

머신러닝은 컴퓨터가 명시적인 프로그래밍 없이 데이터로부터 학습하여 특정 작업을 수행하는 기술입니다. 이를 통해 컴퓨터는 패턴을 찾아내고, 예측이나 결정을 내리며, 경험을 통해 성능을 스스로 개선합니다.


### 체인 재사용

한 번 정의한 체인은 다양한 입력으로 재사용할 수 있습니다.
템플릿의 변수만 바꾸면 됩니다.

아래 코드는 동일한 체인으로 여러 주제를 처리하는 모습을 보여줍니다.

In [ ]:
# 다양한 주제로 동일 체인 재사용
topics = ["블록체인", "클라우드 컴퓨팅", "사이버 보안"]

for topic in topics:
    result = chain.invoke({"topic": topic, "num": 3})
    print(f"[{topic}] {result}", flush = True)
    print()

[블록체인] 블록체인은 거래 기록을 블록 단위로 묶어 체인처럼 연결한 분산형 데이터베이스입니다. 각 블록은 암호화 방식으로 이전 블록과 연결되어 있어, 한 번 기록된 데이터는 위변조하기 매우 어렵습니다. 중앙 관리자 없이 모든 참여자가 기록을 공유하고 검증하여, 투명하고 신뢰할 수 있는 시스템을 제공합니다.

[클라우드 컴퓨팅] 클라우드 컴퓨팅은 인터넷을 통해 서버, 스토리지, 데이터베이스 등 다양한 컴퓨팅 자원을 필요할 때마다 제공하는 서비스 모델입니다. 이를 통해 사용자는 자체 물리적 인프라를 구축할 필요 없이 필요한 만큼만 자원을 유연하게 사용할 수 있습니다. 따라서 비용 절감, 빠른 확장성, 유연성 등의 이점을 제공하여 많은 기업과 개인에게 효율적인 컴퓨팅 환경을 제공합니다.

[사이버 보안] 사이버 보안은 컴퓨터 시스템, 네트워크 및 데이터를 디지털 공격과 무단 접근으로부터 보호하는 활동입니다. 이는 정보, 장치 및 사용자 개인 정보 보호를 위한 다양한 기술과 프로세스를 포함합니다. 궁극적으로 사이버 보안은 연결된 세상에서 디지털 자산의 기밀성, 무결성 및 가용성을 보장하는 것을 목표로 합니다.



### 여러 변수를 사용하는 템플릿

템플릿에는 여러 개의 변수를 사용할 수 있습니다.
`{변수명}` 형태로 원하는 만큼 추가하면 됩니다.

아래 코드는 두 개의 변수를 사용하는 체인을 보여줍니다.

In [ ]:
# 여러 변수를 사용하는 프롬프트 템플릿
prompt_multi = ChatPromptTemplate.from_template(
    "{language}로 {task}하는 코드를 한 줄로 작성해주세요."
)

chain_multi = prompt_multi | model | output_parser

result = chain_multi.invoke({
    "language": "Python",
    "task": "1부터 10까지 합계를 구",
})
print(result)

네, 1부터 10까지 합계를 구하는 Python 한 줄 코드는 다음과 같습니다:

```python
print(sum(range(1, 11)))
```


### from_messages로 역할별 프롬프트 구성

`from_template()`은 단순 문자열 템플릿이지만,
`from_messages()`를 사용하면 system/human/ai 역할별로 메시지를 구성할 수 있습니다.
이 패턴은 노트북 2(System Prompt)에서 본격적으로 다루지만, 형태를 미리 확인해둡니다.

아래 코드는 `from_messages()`로 역할별 메시지를 포함하는 체인을 보여줍니다.

In [ ]:
# from_messages로 역할별 프롬프트 구성
prompt_with_system = ChatPromptTemplate.from_messages([
    ("system", "당신은 IT 용어를 쉽게 설명하는 도우미입니다. 비유를 활용하여 2문장 이내로 답변하세요."),
    ("human", "{term}이(가) 무엇인지 설명해주세요."),
])

chain_system = prompt_with_system | model | output_parser

result = chain_system.invoke({"term": "API"})
print(result)

API는 레스토랑에서 손님(다른 프로그램)과 주방(서비스) 사이의 웨이터와 같아요. 웨이터가 주문(요청)을 받아 주방에 전달하고 요리된 음식(결과)을 가져다주듯, 프로그램들이 서로 소통하며 정보를 주고받는 방법을 정해줍니다.


### 체인에 .batch() 적용

모델뿐 아니라 체인도 `.batch()`를 지원합니다.
여러 입력을 리스트로 전달하면 한 번에 처리합니다.

아래 코드는 체인의 `.batch()`를 보여줍니다.

In [ ]:
# 체인에 batch 적용
inputs = [
    {"term": "API"},
    {"term": "SDK"},
    {"term": "REST"},
]

results = chain_system.batch(inputs)

for inp, res in zip(inputs, results):
    print(f"[{inp['term']}] {res}")
    print()

[API] API는 레스토랑의 웨이터와 같아요. 손님(다른 프로그램)이 주방(서비스)에 직접 가지 않고도 웨이터(API)에게 주문하면, 원하는 정보나 기능을 전달받을 수 있도록 돕는 통로입니다.

[SDK] SDK는 특정 앱을 만들 때 필요한 도구, 부품, 설명서가 모두 담긴 종합 선물 세트라고 생각하면 돼요. 이 세트를 이용하면 처음부터 모든 것을 만들 필요 없이 쉽고 빠르게 앱을 개발할 수 있습니다.

[REST] REST는 웹에서 정보를 주고받는 표준화된 '주문 방식' 같은 거예요. 마치 손님이 메뉴판(자원)을 보고 '이 음식 주세요'(요청)라고 명확히 말하면, 주방(서버)이 알아서 처리해주는 방식과 비슷하죠.



### 파서 없이 체인 실행

`StrOutputParser`를 빼면 체인의 결과는 `AIMessage` 객체 그대로 반환됩니다.
직접 확인해봅시다.

아래 코드는 파서가 있을 때와 없을 때의 반환 타입 차이를 비교합니다.

In [ ]:
# 파서 있는 체인 vs 없는 체인
chain_with_parser = prompt | model | output_parser
chain_without_parser = prompt | model

result_with = chain_with_parser.invoke({"topic": "딥러닝" ,"num" : "2"})
result_without = chain_without_parser.invoke({"topic": "딥러닝","num" : "2"})

print(f"파서 있음 → 타입: {type(result_with).__name__}, 값: {result_with[:50]}...")
print(f"파서 없음 → 타입: {type(result_without).__name__}, 값: {result_without.content[:50]}...")

파서 있음 → 타입: TextAccessor, 값: 딥러닝은 여러 층의 인공신경망을 사용하여 데이터에서 복잡한 패턴과 특징을 스스로 학습하는 ...
파서 없음 → 타입: AIMessage, 값: 딥러닝은 여러 층으로 이루어진 인공신경망을 사용하여 방대한 양의 데이터에서 복잡한 패턴을 ...


> 핵심: `StrOutputParser`는 `AIMessage`에서 `.content`를 추출하는 역할입니다.
> 후속 코드에서 문자열로 바로 사용해야 할 때 편리합니다.
> 메타데이터(토큰 수 등)가 필요하면 파서를 빼고 `AIMessage`를 직접 다루면 됩니다.

---

### 생각해보기

1. google-genai SDK와 LangChain 중 하나만 배워야 한다면 어떤 것을 선택하겠습니까? 그 이유는 무엇인가요?
2. LangChain이 "모델 교체 용이성"을 장점으로 내세웁니다. 실제로 모델을 교체해야 하는 상황은 어떤 경우일까요?
3. LCEL 체인에서 `prompt | model` 사이에 다른 단계를 끼워넣을 수 있을까요? 어떤 용도로 사용할 수 있을지 생각해보세요.

---

# Part 2 — 실습

이론에서 배운 개념을 직접 코드로 작성해봅니다.
TODO 부분을 채워주세요. 막히면 정답 주석을 확인하세요.

### 실습 1-1: google-genai로 간단한 질문 호출

google-genai SDK를 사용하여 Gemini에게 질문을 보내고 응답을 출력하세요.

**요구사항**
1. `client.models.generate_content()`를 사용
2. 모델: `gemini-2.5-flash`
3. 질문: "파이썬이란 무엇인가요?"
4. 응답 텍스트를 출력
5. 입력 토큰 수와 출력 토큰 수를 각각 출력

In [ ]:
# TODO: google-genai SDK로 질문을 보내고 응답을 출력하세요
response = None  # 이 줄을 수정하세요

# ---------- 정답 ----------
# response = client.models.generate_content(
#     model="gemini-2.5-flash",
#     contents="파이썬이란 무엇인가요?",
# )

# 검증
if response is not None:
    print(f"응답: {response.text}")
    print(f"입력 토큰: {response.usage_metadata.prompt_token_count}")
    print(f"추론 토큰:  {response.usage_metadata.thoughts_token_count}")
    print(f"출력 토큰: {response.usage_metadata.candidates_token_count}")
else:
    print("TODO를 완성해주세요")

응답: 파이썬(Python)은 **쉽고 강력하며 다양한 분야에서 사용되는 고급 프로그래밍 언어**입니다.

조금 더 자세히 설명해 드릴게요.

---

### 파이썬이란 무엇인가요?

1.  **고급 프로그래밍 언어 (High-level Programming Language):**
    *   컴퓨터가 이해하는 기계어(0과 1)에 가깝지 않고, 사람이 일상적으로 사용하는 언어(영어)와 유사하게 설계되어 배우기 쉽고 코드를 작성하기 편리합니다.

2.  **인터프리터 언어 (Interpreted Language):**
    *   코드를 작성하면 별도의 '컴파일' 과정 없이 바로 실행될 수 있습니다. (자바나 C++ 같은 언어는 실행 전에 컴파일 과정을 거쳐야 합니다.) 이는 개발 속도를 빠르게 해줍니다.

3.  **객체 지향 프로그래밍 언어 (Object-Oriented Programming Language):**
    *   현실 세계의 사물을 객체(Object)라는 개념으로 모델링하여 프로그램을 만들 수 있게 해줍니다. 코드를 재사용하고 유지보수하기 쉽게 만듭니다.

4.  **범용 프로그래밍 언어 (General-purpose Programming Language):**
    *   특정 목적에만 사용되는 것이 아니라, 웹 개발, 데이터 분석, 인공지능, 자동화, 게임 개발 등 매우 다양한 분야에서 활용될 수 있습니다.

### 파이썬의 주요 특징과 장점

*   **배우기 쉬운 문법:** 사람이 읽고 쓰기 쉬운 간결하고 명확한 문법을 가지고 있어 프로그래밍 초보자에게 특히 인기가 많습니다.
*   **다양한 라이브러리 및 프레임워크:** 웹 개발(Django, Flask), 데이터 과학(NumPy, Pandas, SciPy), 머신러닝/AI(TensorFlow, PyTorch, scikit-learn), GUI 개발(Tkinter, PyQt) 등 특정 작업을 위한 방대한 양의 라이브러리와 프레임워크를 제공하여 개발 시간을 단축시킵니다.
*   

### 실습 1-2: config를 사용한 google-genai 호출

`config` 파라미터를 활용하여 모델의 동작을 제어해보세요.

**요구사항**
1. `client.models.generate_content()`를 사용
2. 모델: `gemini-2.5-flash`
3. 질문: "대한민국의 역대 대통령을 나열해주세요."
4. `config`에서 `max_output_tokens`를 `50`으로 제한
5. `config`에서 `temperature`를 `0.0`으로 설정
6. 응답 텍스트와 출력 토큰 수를 출력 (50 토큰 제한이 적용되었는지 확인)

In [ ]:
# TODO: config 파라미터를 사용하여 호출하세요
response_limited = None  # 이 줄을 수정하세요

# ---------- 정답 ----------
# response_limited = client.models.generate_content(
#     model="gemini-2.5-flash",
#     contents="대한민국의 역대 대통령을 나열해주세요.",
#     config={
#         "max_output_tokens": 50,
#         "temperature": 0.0,
#     },
# )
# candidate = response_limited.candidates[0]

# 검증
if response_limited is not None:
    print(f"응답: {response_limited.text}")
    print(f"출력 토큰 수: {response_limited.usage_metadata.candidates_token_count}")
    print("(응답이 중간에 끊겼다면 max_output_tokens 제한이 적용된 것입니다)")
else:
    print("TODO를 완성해주세요")
print(f"Stop Reason: {candidate.finish_reason}")

응답: 대한민
출력 토큰 수: 2
(응답이 중간에 끊겼다면 max_output_tokens 제한이 적용된 것입니다)
Stop Reason: FinishReason.MAX_TOKENS


### 실습 1-3: LangChain으로 동일 질문 호출 후 반환 객체 비교

LangChain의 `ChatGoogleGenerativeAI`를 사용하여 질문을 보내고,
google-genai와 반환 객체의 차이를 정리하세요.

**요구사항**
1. `ChatGoogleGenerativeAI`로 모델 생성 (모델: `gemini-2.5-flash`)
2. `.invoke()`로 호출 (질문: "파이썬이란 무엇인가요?")
3. 반환 객체의 타입을 출력
4. 텍스트 내용(앞 200자)을 출력
5. `response_metadata`의 키 목록을 출력
6. `usage_metadata`에서 입력/출력 토큰 수를 출력

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# TODO 1: LangChain 모델 생성
lc_model = None  # 이 줄을 수정하세요

# TODO 2: invoke로 호출
lc_response = None  # 이 줄을 수정하세요

# ---------- 정답 ----------
# lc_model = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash",
#     api_key=GEMINI_API_KEY,
# )
# lc_response = lc_model.invoke("파이썬이란 무엇인가요?")

# 검증
if lc_response is not None:
    print(f"타입: {type(lc_response).__name__}")
    print(f"텍스트: {lc_response.content[:200]}...")
    print(f"\nresponse_metadata 키: {list(lc_response.response_metadata.keys())}")
    print(f"\n입력 토큰: {lc_response.usage_metadata['input_tokens']}")
    print(f"출력 토큰: {lc_response.usage_metadata['output_tokens']}")
else:
    print("TODO를 완성해주세요")

타입: AIMessage
텍스트: 파이썬(Python)은 오늘날 가장 인기 있고 널리 사용되는 **고급(High-level), 인터프리터 방식(Interpreted), 다목적(General-purpose) 프로그래밍 언어**입니다.

간단히 말해, 컴퓨터에게 어떤 작업을 시킬지 명령을 내리는 도구이며, 그 방식이 인간의 언어와 가까워 배우고 사용하기 쉽다는 특징을 가지고 있습니다.

###...

response_metadata 키: ['finish_reason', 'model_name', 'safety_ratings', 'model_provider']

입력 토큰: 9
출력 토큰: 1852


### 실습 1-4: LangChain 메시지 객체로 호출

문자열 대신 LangChain 메시지 객체를 직접 생성하여 모델을 호출해보세요.

**요구사항**
1. `SystemMessage`로 시스템 메시지 생성: "당신은 역사 전문가입니다. 간결하게 답변하세요."
2. `HumanMessage`로 사용자 메시지 생성: "조선의 마지막 왕은 누구인가요?"
3. 두 메시지를 리스트로 묶어 `model.invoke()`에 전달
4. 응답 내용을 출력

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

# TODO 1: SystemMessage 생성
system_msg = None  # 이 줄을 수정하세요

# TODO 2: HumanMessage 생성
human_msg = None  # 이 줄을 수정하세요

# TODO 3: 메시지 리스트로 invoke 호출
msg_response = None  # 이 줄을 수정하세요

# ---------- 정답 ----------
# system_msg = SystemMessage(content="당신은 역사 전문가입니다. 간결하게 답변하세요.")
# human_msg = HumanMessage(content="조선의 마지막 왕은 누구인가요?")
# msg_response = model.invoke([system_msg, human_msg])

# 검증
if msg_response is not None:
    print(f"응답 타입: {type(msg_response).__name__}")
    print(f"응답: {msg_response.content}")
else:
    print("TODO를 완성해주세요")

### 실습 1-5: LCEL 체인 구성 (단일 변수)

`PromptTemplate | ChatModel | StrOutputParser` 구조의 LCEL 체인을 직접 구성하세요.

**요구사항**
1. `ChatPromptTemplate.from_template()`으로 프롬프트 템플릿 생성
   - 변수: `{language}`
   - 템플릿: "{language} 프로그래밍 언어의 대표적인 사용 사례를 3가지 알려주세요."
2. `StrOutputParser`로 출력 파서 생성
3. `|`로 체인 구성: `prompt | model | parser`
4. `"Java"`를 입력으로 체인 실행
5. 결과와 반환 타입을 출력

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# TODO 1: 프롬프트 템플릿 생성
prompt = None  # 이 줄을 수정하세요

# TODO 2: 출력 파서 생성
parser = None  # 이 줄을 수정하세요

# TODO 3: 체인 구성
chain = None  # 이 줄을 수정하세요

# ---------- 정답 ----------
# prompt = ChatPromptTemplate.from_template(
#     "{language} 프로그래밍 언어의 대표적인 사용 사례를 3가지 알려주세요."
# )
# parser = StrOutputParser()
# chain = prompt | model | parser

# 검증
if chain is not None:
    result = chain.invoke({"language": "Java"})
    print(result)
    print(f"\n반환 타입: {type(result).__name__}")
else:
    print("TODO를 완성해주세요")

자바(Java)는 매우 광범위하게 사용되는 프로그래밍 언어이며, 특히 안정성, 확장성, 성능이 중요한 분야에서 강점을 보입니다. 대표적인 사용 사례 3가지는 다음과 같습니다.

1.  **기업용 애플리케이션 및 웹 서비스 (Enterprise Applications & Web Services)**
    자바는 강력한 안정성, 확장성, 보안성을 바탕으로 대규모 기업 시스템 구축에 가장 널리 사용됩니다. 금융권 시스템, 전사적 자원 관리(ERP), 고객 관계 관리(CRM), 물류 시스템 등 복잡하고 중요한 백엔드 시스템 개발에 주로 활용됩니다. 또한 Spring, Spring Boot, Jakarta EE(이전 J2EE)와 같은 프레임워크를 통해 RESTful API, 마이크로서비스 등 현대적인 웹 서비스 개발에도 핵심적인 역할을 합니다.

2.  **안드로이드 모바일 애플리케이션 개발 (Android Mobile Application Development)**
    안드로이드 운영체제는 자바를 기반으로 개발되었으며, 오랫동안 안드로이드 앱 개발의 공식 언어였습니다. 비록 최근에는 Kotlin이 대세로 떠오르고 있지만, 수많은 기존 앱과 라이브러리가 자바로 작성되어 있으며, 여전히 안드로이드 개발에서 중요한 위치를 차지하고 있습니다. 전 세계 수십억 대의 안드로이드 기기에서 실행되는 앱들이 자바로 만들어졌거나 자바와 깊은 연관을 가지고 있습니다.

3.  **빅데이터, 클라우드 및 대규모 백엔드 시스템 (Big Data, Cloud, and Large-scale Backend Systems)**
    자바는 대용량 데이터를 처리하고 분산 시스템을 구축하는 데 매우 적합합니다. Apache Hadoop, Apache Spark, Apache Kafka, Elasticsearch 등 많은 빅데이터 기술 스택이 자바를 기반으로 하거나 자바로 구현되어 있습니다. 또한 클라우드 환경에서 동작하는 고성능의 마이크로서비스 아키텍처나 서버리스 애플리케이션의 백엔드 개발에도

### 실습 1-6: 여러 변수 + from_messages 체인 구성

system 메시지를 포함하고 여러 변수를 사용하는 LCEL 체인을 구성하세요.

**요구사항**
1. `ChatPromptTemplate.from_messages()`를 사용
2. system 메시지: "당신은 {role}입니다. 핵심만 간결하게 답변하세요."
3. human 메시지: "{question}"
4. `StrOutputParser`로 체인 구성
5. `role="요리 전문가"`, `question="된장찌개 끓이는 순서를 알려주세요"` 로 실행
6. 같은 체인으로 `role="체육 교사"`, `question="스트레칭의 중요성을 알려주세요"` 도 실행

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# TODO 1: from_messages로 프롬프트 템플릿 생성
prompt_role = None  # 이 줄을 수정하세요

# TODO 2: 체인 구성
chain_role = None  # 이 줄을 수정하세요

# ---------- 정답 ----------
# prompt_role = ChatPromptTemplate.from_messages([
#     ("system", "당신은 {role}입니다. 핵심만 간결하게 답변하세요."),
#     ("human", "{question}"),
# ])
# chain_role = prompt_role | model | StrOutputParser()

# 검증
if chain_role is not None:
    # 첫 번째 호출
    result1 = chain_role.invoke({
        "role": "요리 전문가",
        "question": "된장찌개 끓이는 순서를 알려주세요",
    })
    print("[요리 전문가]")
    print(result1)
    print()

    # 두 번째 호출 — 같은 체인, 다른 입력
    result2 = chain_role.invoke({
        "role": "체육 교사",
        "question": "스트레칭의 중요성을 알려주세요",
    })
    print("[체육 교사]")
    print(result2)
else:
    print("TODO를 완성해주세요")

[요리 전문가]
1.  **육수를 끓여 된장을 푼다.** (멸치 다시마 육수가 기본)
2.  **단단한 재료(감자, 무 등)를 먼저 넣고 끓인다.**
3.  **이어서 호박, 양파, 두부를 넣는다.**
4.  **마지막으로 버섯, 파, 고추, 다진 마늘을 넣고 한소끔 더 끓여 마무리한다.**

[체육 교사]
스트레칭, 왜 중요할까요? 딱 세 가지로 요약해 드릴게요!

1.  **몸이 유연해져요:** 움직임이 부드러워지고, 활동 범위가 넓어져요.
2.  **부상을 예방해요:** 근육과 관절을 보호해서 다칠 위험을 줄여줍니다.
3.  **피로 회복에 좋아요:** 운동 후 근육통을 줄여주고, 몸의 긴장을 풀어줍니다.

매일 꾸준히 스트레칭해서 건강한 몸을 만들어요!


### 실습 1-7: 체인 batch 실행

실습 1-5에서 만든 체인(또는 동일 구조의 체인)을 `.batch()`로 실행해보세요.

**요구사항**
1. 프롬프트 변수 `{language}`를 사용하는 체인 구성 (실습 1-5의 체인 재활용 가능)
2. 입력: `[{"language": "Python"}, {"language": "JavaScript"}, {"language": "Rust"}]`
3. `.batch()`로 3개를 한 번에 실행
4. 각 결과를 `[언어명] 결과` 형태로 출력

In [ ]:
# TODO 1: 체인 구성 (실습 1-5의 chain을 재사용하거나 새로 만드세요)
batch_chain = None  # 이 줄을 수정하세요

# TODO 2: batch 입력 리스트 구성
batch_inputs = None  # 이 줄을 수정하세요

# ---------- 정답 ----------
# batch_chain = (
#     ChatPromptTemplate.from_template(
#         "{language} 프로그래밍 언어의 대표적인 사용 사례를 3가지 알려주세요."
#     )
#     | model
#     | StrOutputParser()
# )
# batch_inputs = [
#     {"language": "Python"},
#     {"language": "JavaScript"},
#     {"language": "Rust"},
# ]

# 검증
if batch_chain is not None and batch_inputs is not None:
    results = batch_chain.batch(batch_inputs)
    for inp, res in zip(batch_inputs, results):
        print(f"[{inp['language']}]")
        print(res)
        print()
else:
    print("TODO를 완성해주세요")

[Python]
파이썬은 다재다능하고 강력한 언어로, 다양한 분야에서 널리 사용됩니다. 그 중 가장 대표적인 3가지 사용 사례는 다음과 같습니다.

1.  **데이터 과학 및 인공지능 (AI) / 머신러닝**
    *   **설명:** 파이썬은 데이터 분석, 시각화, 머신러닝 모델 개발, 딥러닝 구현 등에 가장 널리 사용되는 언어입니다. 방대한 양의 데이터를 처리하고 패턴을 찾아내며, 예측 모델을 만드는 데 최적화되어 있습니다.
    *   **장점:**
        *   **풍부한 라이브러리:** NumPy, Pandas (데이터 처리 및 분석), Matplotlib, Seaborn (데이터 시각화), Scikit-learn (머신러닝), TensorFlow, PyTorch (딥러닝) 등 강력하고 성숙한 라이브러리 생태계를 갖추고 있습니다.
        *   **쉬운 문법:** 복잡한 알고리즘을 간결하고 직관적인 코드로 구현할 수 있어 개발 생산성이 높습니다.
    *   **예시:** 추천 시스템 개발 (넷플릭스, 유튜브), 의료 영상 분석, 자연어 처리, 자율 주행 자동차 알고리즘 개발 등.

2.  **웹 개발 (백엔드)**
    *   **설명:** 파이썬은 웹 애플리케이션의 서버 측 로직(백엔드)을 개발하는 데 매우 인기가 많습니다. 사용자의 요청을 처리하고, 데이터베이스와 상호작용하며, 동적인 웹 페이지를 생성하는 역할을 합니다.
    *   **장점:**
        *   **강력한 프레임워크:** Django와 Flask와 같은 강력하고 성숙한 웹 프레임워크를 제공하여 빠르고 안정적인 웹 애플리케이션 개발을 가능하게 합니다.
        *   **높은 생산성:** 간결한 문법과 잘 정리된 구조 덕분에 개발 속도가 빠르며 유지보수가 용이합니다.
        *   **확장성:** 대규모 서비스에 적합한 구조를 갖추고 있습니다.
    *   **예시:** 인스타그램, 스포티파이, 드롭박스 등 많은 대형 서비스들이 파이썬을

---

### 생각해보기

1. 실습 1-1과 1-3에서 동일한 질문을 했지만 응답 내용이 다를 수 있습니다. 왜 그럴까요? (힌트: temperature 기본값)
2. LCEL 체인에서 `StrOutputParser` 대신 다른 파서를 사용하면 어떤 가능성이 열릴까요? (예: JSON 파싱, Pydantic 파싱 등)
3. 실습 1-6에서 system 메시지의 `role`을 바꾸면 같은 질문에도 다른 관점의 답변이 나옵니다. 이 특성을 실무에서 어떻게 활용할 수 있을까요?

---

# Part 3 — 챌린지

이론과 실습에서 배운 내용을 응용하여 스스로 설계하고 구현합니다.
정답이 제공되지 않으며, 강사와 함께 진행합니다.

### 챌린지 1-1: 두 방식의 응답 시간 측정 후 비교 분석 (난이도: ★☆☆)

> 이 챌린지는 정답이 제공되지 않습니다. 강사와 함께 진행하세요.

google-genai 직접 호출과 LangChain 호출의 응답 시간을 측정하고 비교해보세요.

**조건**
- 동일한 프롬프트를 사용할 것
- 각 방식을 3회 반복 호출하여 평균 시간을 계산할 것
- `time` 모듈을 활용할 것

**분석 관점**
- 두 방식의 응답 시간에 차이가 있는가?
- 차이가 있다면 왜 그런가? (힌트: LangChain은 래퍼입니다)
- 이 차이가 실무에서 의미 있는 수준인가?

**힌트**
- `time.time()`으로 호출 전후 시간을 측정하세요
- 첫 번째 호출은 워밍업 효과가 있을 수 있으니 제외를 고려하세요
- 결과를 표 형태로 정리하면 비교하기 쉽습니다

In [ ]:
import time
import os
from google import genai
from langchain_google_genai import ChatGoogleGenerativeAI

PROMPT = "대한민국의 수도를 한 문장으로 알려주세요."

# 클라이언트를 전역에서 한 번만 초기화
client = genai.Client(api_key=GEMINI_API_KEY)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=GEMINI_API_KEY)


def measure_time(func, repeat=3, warmup=True):
    times = []
    total_repeat = repeat + 1 if warmup else repeat

    for i in range(total_repeat):
        start = time.time()
        result = func()
        elapsed = time.time() - start

        if warmup and i == 0:
            print(f"  [warmup] {elapsed:.3f}s (제외)")
            continue

        run_num = i if warmup else i + 1
        times.append(elapsed)
        print(f"  [{run_num}회] {elapsed:.3f}s")

    avg = sum(times) / len(times)
    print(f"  → 평균: {avg:.3f}s\n")
    return avg


def call_genai_direct() -> str:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=PROMPT,
    )
    return response.text


def call_langchain() -> str:
    response = llm.invoke(PROMPT)
    return response.content


# ── 실행 ─────────────────────────────────────────────────────
print("=" * 45)
print("▶ google-genai 직접 호출")
print("=" * 45)
direct_avg = measure_time(call_genai_direct)

print("=" * 45)
print("▶ LangChain 호출")
print("=" * 45)
lc_avg = measure_time(call_langchain)

# ── 비교 표 ───────────────────────────────────────────────────
diff     = lc_avg - direct_avg
overhead = (diff / direct_avg) * 100

print("=" * 45)
print("📊 비교 결과")
print("=" * 45)
print(f"{'방식':<20} {'평균 응답시간':>12}")
print("-" * 35)
print(f"{'google-genai 직접':<20} {direct_avg:>10.3f}s")
print(f"{'LangChain 래퍼':<20} {lc_avg:>10.3f}s")
print("-" * 35)
print(f"{'오버헤드':<20} {diff:>+10.3f}s  ({overhead:+.1f}%)")

▶ google-genai 직접 호출
str
  [warmup] 1.402s (제외)
str
  [1회] 1.217s
str
  [2회] 1.261s
str
  [3회] 1.926s
  → 평균: 1.468s

▶ LangChain 호출
str
  [warmup] 0.794s (제외)
str
  [1회] 1.898s
str
  [2회] 1.961s
str
  [3회] 0.754s
  → 평균: 1.538s

📊 비교 결과
방식                        평균 응답시간
-----------------------------------
google-genai 직접           1.468s
LangChain 래퍼              1.538s
-----------------------------------
오버헤드                     +0.070s  (+4.7%)


---

### 생각해보기

1. LangChain이 google-genai보다 약간 느리다면, 그럼에도 LangChain을 선택해야 하는 상황은 어떤 경우일까요?
2. 네트워크 지연이 큰 환경(해외 서버 등)에서는 두 방식의 시간 차이가 더 벌어질까요, 줄어들까요?
3. 응답 시간 외에 두 방식을 비교할 수 있는 다른 성능 지표로는 무엇이 있을까요?